# CatVTON 가상 착용 테스트 (AI 옷장)

사람 사진에 옷 사진을 입혀 보는 공개 모델 [CatVTON](https://github.com/Zheng-Chong/CatVTON)을 시험하는 노트북이에요.

1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 를 고르고 저장해요.
2. 위에서부터 셀을 차례로 실행해요. 처음 한 번은 모델 파일(약 6GB)을 받느라 몇 분 걸려요.
3. 준비물
   - 사람 사진: 정면, 허리 아래까지 보이게, 팔을 몸에서 살짝 뗀 사진
   - 옷 사진: 옷만 펼쳐 찍은 사진일수록 잘 나와요 (상의·하의·원피스. 신발은 안 돼요)

알아 둘 점
- 결과는 AI가 그린 예시 이미지예요. 실제 사이즈와 핏은 반영되지 않아요.
- CatVTON은 비상업 라이선스(CC BY-NC-SA 4.0)라 캡스톤 시연·연구용으로만 써요.
- 올린 사진은 이 Colab 안에서만 처리돼요. 런타임을 끄면 함께 지워져요.

In [ ]:
# 1. GPU 확인
import torch
assert torch.cuda.is_available(), "GPU가 꺼져 있어요. 메뉴 런타임 → 런타임 유형 변경 → T4 GPU 를 고르고 다시 실행하세요."
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f}GB")

In [ ]:
# 2. CatVTON 코드 받기 + 패키지 설치 (2025-12 버전으로 고정, 여기서 시험한 버전 그대로)
import os
CATVTON_DIR = "/content/CatVTON"
if not os.path.exists(CATVTON_DIR):
    !git clone -q https://github.com/Zheng-Chong/CatVTON.git {CATVTON_DIR}
!git -C {CATVTON_DIR} checkout -q 7818397
!pip install -q diffusers==0.41.0 transformers==5.19.0 accelerate==1.15.0 huggingface_hub==1.33.0 fvcore==0.1.5.post20221221 cloudpickle==3.1.2 omegaconf==2.4.0 pycocotools==2.0.11 av==18.1.0
# Colab에 미리 깔린 torchao(양자화 도구)는 CatVTON에 필요 없고, diffusers가 불러오다 충돌할 수 있어 지운다
!pip uninstall -y -q torchao

In [ ]:
# 3. 모델 불러오기 (처음엔 다운로드 때문에 몇 분)
import os, sys, time
if "torchao" in sys.modules:   # 지운 torchao가 메모리에 남아 있으면 diffusers가 계속 실패함
    raise RuntimeError("메뉴 런타임 → 세션 다시 시작 을 누른 뒤 1번 셀부터 다시 실행해 주세요.")
sys.path.insert(0, CATVTON_DIR)
import torch
from huggingface_hub import snapshot_download
from diffusers.image_processor import VaeImageProcessor
import detectron2.config.defaults as d2_defaults
from model.cloth_masker import AutoMasker, vis_mask
from model.pipeline import CatVTONPipeline
from utils import resize_and_crop, resize_and_padding

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
d2_defaults._C.MODEL.DEVICE = DEVICE          # 몸 부위를 찾는 DensePose도 같은 장치에서
if not all(k in globals() for k in ("pipeline", "automasker", "mask_blur")):   # 다시 실행해도 두 번 올리지 않게
    t0 = time.time()
    cat_dir = snapshot_download("zhengchong/CatVTON", allow_patterns=["DensePose/*", "SCHP/*", "mix-48k-1024/*"])
    base_dir = snapshot_download("booksforcharlie/stable-diffusion-inpainting",
                                 allow_patterns=["model_index.json", "scheduler/*", "unet/*", "feature_extractor/*", "safety_checker/*"])
    pipeline = CatVTONPipeline(base_ckpt=base_dir, attn_ckpt=cat_dir, attn_ckpt_version="mix",
                               weight_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
                               use_tf32=True, device=DEVICE)
    automasker = AutoMasker(densepose_ckpt=os.path.join(cat_dir, "DensePose"), schp_ckpt=os.path.join(cat_dir, "SCHP"),
                            device=DEVICE)
    mask_blur = VaeImageProcessor(vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True)
    print(f"모델 준비 완료 ({time.time() - t0:.0f}초)")

In [ ]:
# 4. 입혀 보기 함수
import numpy as np
from PIL import Image, ImageDraw, ImageOps
from IPython.display import display

KINDS = {"상의": "upper", "하의": "lower", "원피스·전신": "overall", "아우터": "outer", "이너": "inner"}
SIZES = {"보통 (768x1024)": (768, 1024), "빠르게 (384x512)": (384, 512)}
WORK = "/content/tryon"
for d in ("person", "cloth", "results"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)


class TryOnError(Exception):   # 사용자에게 그대로 보여 줄 안내
    pass


pipeline.skip_safety_check = True   # 안전 필터는 try_on 안에서 직접 돌려, 영어 경고 그림 대신 알아듣기 쉬운 안내로


def open_image(f):
    return ImageOps.exif_transpose(Image.open(f)).convert("RGB")   # 폰 사진의 회전 정보 반영


def try_on(person, cloth, cloth_type="upper", steps=50, guidance=2.5, seed=42, size=(768, 1024)):
    """사람 사진(PIL)에 옷 사진(PIL)을 입힌다. 반환 {"result", "masked", "person", "cloth", "mask_s", "tryon_s"}"""
    person = resize_and_crop(person, size)        # 3:4로 가운데 자르기
    cloth = resize_and_padding(cloth, size)       # 3:4로 여백 채우기
    t = time.time()
    parts = automasker(person, cloth_type)
    if (np.array(parts["densepose"]) > 0).mean() < 0.005:      # 몸 부위가 거의 안 잡히면 사람이 없는 사진
        raise TryOnError("사진에서 사람을 찾지 못했어요. 몸이 잘 보이게 다시 찍어 주세요.")
    mask = mask_blur.blur(parts["mask"], blur_factor=9)
    mask_s = time.time() - t
    t = time.time()
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    result = pipeline(image=person, condition_image=cloth, mask=mask, num_inference_steps=steps,
                      guidance_scale=guidance, generator=gen, width=size[0], height=size[1])[0]
    if pipeline.run_safety_checker(image=np.array(result)[None])[1][0]:
        raise TryOnError("결과가 안전 필터에 걸렸어요. 다른 사진으로 다시 해 보세요.")
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    return {"result": result, "masked": vis_mask(person, mask), "person": person, "cloth": cloth,
            "mask_s": mask_s, "tryon_s": time.time() - t}


def side_by_side(images, labels=None, height=420):
    ims = [im.resize((round(im.width * height / im.height), height)) for im in images]
    pad = 24 if labels else 0
    out = Image.new("RGB", (sum(i.width for i in ims) + 8 * (len(ims) - 1), height + pad), "white")
    draw, x = ImageDraw.Draw(out), 0
    for i, im in enumerate(ims):
        out.paste(im, (x, pad))
        if labels:
            draw.text((x + 4, 5), labels[i], fill="black")    # 기본 글꼴은 한글이 안 돼서 영어로
        x += im.width + 8
    return out


def files_in(folder):
    return sorted(f for f in os.listdir(f"{WORK}/{folder}") if not f.startswith("."))

## 사진 준비
사람 사진과 옷 사진을 올려요. 옷 사진 파일 이름 앞에 `상의_`, `하의_`, `원피스_`, `아우터_` 를 붙이면 아래 '한꺼번에' 셀이 종류를 알아서 골라요 (예: `상의_흰셔츠.jpg`).

In [ ]:
# 5. 사진 올리기 (여러 장 골라도 돼요)
from google.colab import files
print("① 사람 사진을 고르세요")
for name, data in files.upload().items():
    open(f"{WORK}/person/{name}", "wb").write(data)
print("② 옷 사진을 고르세요")
for name, data in files.upload().items():
    open(f"{WORK}/cloth/{name}", "wb").write(data)
print("사람:", files_in("person"))
print("옷:", files_in("cloth"))

In [ ]:
# 5-1. (선택) 노트북 카메라로 사람 사진 찍기 — 버튼을 누르면 3초 뒤에 찍혀요
from base64 import b64decode
from IPython.display import Javascript
from google.colab.output import eval_js

display(Javascript('''
async function capture() {
  const box = document.createElement("div"), video = document.createElement("video"), btn = document.createElement("button");
  btn.textContent = "3초 뒤 찍기";
  video.style.maxWidth = "360px"; video.style.transform = "scaleX(-1)"; video.playsInline = true;
  box.append(video, document.createElement("br"), btn);
  document.body.appendChild(box);
  const stream = await navigator.mediaDevices.getUserMedia({video: {width: {ideal: 1280}, height: {ideal: 1280}}});
  video.srcObject = stream;
  await video.play();
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
  await new Promise((r) => (btn.onclick = r));
  for (let s = 3; s > 0; s--) { btn.textContent = s + "..."; await new Promise((r) => setTimeout(r, 1000)); }
  const c = document.createElement("canvas");
  c.width = video.videoWidth; c.height = video.videoHeight;
  c.getContext("2d").drawImage(video, 0, 0);
  stream.getTracks().forEach((t) => t.stop());
  box.remove();
  return c.toDataURL("image/jpeg", 0.92);
}'''))
shot = f"camera_{time.strftime('%H%M%S')}.jpg"
open(f"{WORK}/person/{shot}", "wb").write(b64decode(eval_js("capture()").split(",")[1]))
im = open_image(f"{WORK}/person/{shot}")
display(im.resize((360, round(im.height * 360 / im.width))))
print("저장:", shot)

## 입혀 보기

In [ ]:
#@title 6. 한 장 입혀 보기 (비워 두면 첫 번째 사진)
person_file = ""  #@param {type:"string"}
cloth_file = ""  #@param {type:"string"}
cloth_kind = "상의"  #@param ["상의", "하의", "원피스·전신", "아우터", "이너"]
steps = 50  #@param {type:"slider", min:10, max:50, step:5}
size_name = "보통 (768x1024)"  #@param ["보통 (768x1024)", "빠르게 (384x512)"]
seed = 42  #@param {type:"integer"}

assert files_in("person") and files_in("cloth"), "5번 셀에서 사람 사진과 옷 사진을 먼저 올려 주세요."
person_file = person_file or files_in("person")[0]
cloth_file = cloth_file or files_in("cloth")[0]
out = try_on(open_image(f"{WORK}/person/{person_file}"), open_image(f"{WORK}/cloth/{cloth_file}"),
             KINDS[cloth_kind], steps=steps, seed=seed, size=SIZES[size_name])
display(side_by_side([out["person"], out["masked"], out["cloth"], out["result"]],
                     ["person", "area to repaint", "cloth", "result"]))
save = f"{WORK}/results/{os.path.splitext(person_file)[0]}__{os.path.splitext(cloth_file)[0]}_{steps}.jpg"
out["result"].save(save, quality=92)
print(f"몸 영역 찾기 {out['mask_s']:.1f}초 + 입히기 {out['tryon_s']:.1f}초 ({steps}단계, {size_name}) → {save}")

In [ ]:
#@title 7. 한꺼번에: 모든 사람 사진 × 모든 옷 사진 (종류는 파일 이름 앞부분으로)
steps = 30  #@param {type:"slider", min:10, max:50, step:5}
PREFIX = {"상의": "upper", "하의": "lower", "원피스": "overall", "아우터": "outer", "이너": "inner"}


def kind_of(name):
    return next((v for k, v in PREFIX.items() if name.startswith(k)), "upper")


assert files_in("person") and files_in("cloth"), "5번 셀에서 사람 사진과 옷 사진을 먼저 올려 주세요."
rows = []
for p in files_in("person"):
    person = open_image(f"{WORK}/person/{p}")
    for c in files_in("cloth"):
        out = try_on(person, open_image(f"{WORK}/cloth/{c}"), kind_of(c), steps=steps)
        out["result"].save(f"{WORK}/results/{os.path.splitext(p)[0]}__{os.path.splitext(c)[0]}_{steps}.jpg", quality=92)
        print(f"{p} + {c} ({kind_of(c)}): {out['mask_s'] + out['tryon_s']:.1f}초")
        display(side_by_side([out["person"], out["cloth"], out["result"]], height=320))
        rows.append(out["mask_s"] + out["tryon_s"])
if rows:
    print(f"{len(rows)}장, 한 장 평균 {sum(rows) / len(rows):.1f}초")

In [ ]:
#@title 8. 단계 수에 따른 시간·품질 비교 (6번에서 고른 사진으로)
compare_steps = [10, 20, 30, 50]
outs, labels = [], []
for s in compare_steps:
    o = try_on(open_image(f"{WORK}/person/{person_file}"), open_image(f"{WORK}/cloth/{cloth_file}"),
               KINDS[cloth_kind], steps=s, seed=seed)
    outs.append(o["result"])
    labels.append(f"{s} steps  {o['mask_s'] + o['tryon_s']:.1f}s")
display(side_by_side(outs, labels, height=380))

In [ ]:
#@title 9. 코디 전체 입혀 보기 (상의 → 하의 차례로 두 번)
person_file2 = ""  #@param {type:"string"}
top_file = ""  #@param {type:"string"}
bottom_file = ""  #@param {type:"string"}
steps = 30  #@param {type:"slider", min:10, max:50, step:5}

assert top_file and bottom_file, "상의 사진과 하의 사진 파일 이름을 적어 주세요."
assert files_in("person") and files_in("cloth"), "5번 셀에서 사람 사진과 옷 사진을 먼저 올려 주세요."
person = open_image(f"{WORK}/person/{person_file2 or files_in('person')[0]}")
first = try_on(person, open_image(f"{WORK}/cloth/{top_file}"), "upper", steps=steps)
second = try_on(first["result"], open_image(f"{WORK}/cloth/{bottom_file}"), "lower", steps=steps)
display(side_by_side([first["person"], first["cloth"], second["cloth"], first["result"], second["result"]],
                     ["person", "top", "bottom", "top only", "top + bottom"], height=360))
total = first["mask_s"] + first["tryon_s"] + second["mask_s"] + second["tryon_s"]
print(f"전체 {total:.1f}초")

In [ ]:
# 10. 결과 사진 한꺼번에 내려받기
import shutil
shutil.make_archive("/content/tryon_results", "zip", f"{WORK}/results")
files.download("/content/tryon_results.zip")

## 웹페이지 연결 시험
Colab GPU에 가상 착용 서버를 열고, 휴대폰이나 노트북 브라우저로 들어갈 수 있는 시험 페이지 주소를 만들어요.

- 페이지에서 옷 사진을 고르고 **가상 착용(예시)** 을 누르면 작은 창에 카메라 화면이 떠요. **3초 뒤 찍기**를 누르면 찍힌 사진에 옷을 입혀 보여 줘요.
- 휴대폰은 카카오톡 안 브라우저 말고 크롬이나 사파리로 열어야 카메라가 켜져요.
- 주소는 이 Colab이 켜져 있는 동안만 동작해요. 받은 사진은 파일로 저장하지 않고, 결과는 최근 20장만 메모리에 잠깐 들고 있어요.
- 여러 명이 동시에 쓰면 GPU가 한 장씩 차례로 처리해요 (페이지에 대기 인원이 보여요).

In [ ]:
TEST_PAGE = r'''<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>가상 착용 시험</title>
<link rel="icon" href="data:,">
<style>
  :root { --bg: #f6f5f2; --card: #ffffff; --ink: #1d1d1f; --sub: #6b6b70; --line: #e2e0da; --accent: #2f5d50; --on-accent: #ffffff; --warn: #a33a2a; }
  @media (prefers-color-scheme: dark) {
    :root { --bg: #151515; --card: #202020; --ink: #f1f1f1; --sub: #a3a3a8; --line: #343434; --accent: #82c2ab; --on-accent: #10251e; --warn: #f09a8a; }
  }
  * { box-sizing: border-box; }
  body { margin: 0; background: var(--bg); color: var(--ink); font: 16px/1.5 system-ui, -apple-system, "Apple SD Gothic Neo", "Malgun Gothic", sans-serif; }
  main { max-width: 560px; margin: 0 auto; padding: 20px 16px 40px; }
  h1 { font-size: 22px; margin: 0; }
  h2 { font-size: 16px; margin: 0 0 10px; }
  .sub { color: var(--sub); font-size: 14px; margin: 4px 0 0; }
  .card { background: var(--card); border: 1px solid var(--line); border-radius: 14px; padding: 16px; margin-top: 16px; }
  .row { display: flex; flex-wrap: wrap; gap: 8px; align-items: center; }
  button, .btn, select { font: inherit; font-size: 15px; border-radius: 10px; border: 1px solid var(--line); background: var(--card); color: var(--ink); padding: 10px 14px; cursor: pointer; }
  .primary { background: var(--accent); color: var(--on-accent); border-color: var(--accent); font-weight: 600; }
  button:disabled { opacity: .45; cursor: not-allowed; }
  .wide { width: 100%; margin-top: 16px; padding: 14px; }
  .thumb { display: block; max-width: 140px; max-height: 180px; margin-top: 12px; border-radius: 10px; border: 1px solid var(--line); }
  [hidden] { display: none !important; }
  dialog { width: min(92vw, 480px); border: 1px solid var(--line); border-radius: 16px; padding: 16px; background: var(--card); color: var(--ink); }
  dialog::backdrop { background: rgba(0, 0, 0, .55); }
  .stage { position: relative; background: #000; border-radius: 12px; overflow: hidden; aspect-ratio: 3 / 4; display: grid; place-items: center; }
  .stage video, .stage img { width: 100%; height: 100%; object-fit: contain; }
  .mirror { transform: scaleX(-1); }
  .count { position: absolute; font-size: 72px; font-weight: 700; color: #fff; text-shadow: 0 2px 12px rgba(0, 0, 0, .6); }
  .busy { color: #fff; text-align: center; padding: 16px; }
  .note { font-size: 13px; color: var(--sub); margin: 8px 0 0; }
  .err { color: var(--warn); }
  .top { display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px; }
  .top button { padding: 6px 12px; }
  .full { width: 100%; margin-top: 10px; }
  .pair { display: flex; gap: 8px; margin-top: 8px; }
  .pair > * { flex: 1; text-align: center; }
</style>
</head>
<body>
<main>
  <h1>가상 착용 시험</h1>
  <p class="sub">AI 옷장 · CatVTON 연결 시험 페이지 (Colab GPU에서 처리)</p>

  <section class="card">
    <h2>1. 입어 볼 옷</h2>
    <div class="row">
      <label class="btn">옷 사진 고르기<input id="cloth-file" type="file" accept="image/*" hidden></label>
      <select id="kind" aria-label="옷 종류">
        <option value="upper">상의</option>
        <option value="lower">하의</option>
        <option value="overall">원피스·전신</option>
        <option value="outer">아우터 (시험)</option>
      </select>
      <select id="steps" aria-label="품질">
        <option value="20">빠르게 (20단계)</option>
        <option value="30" selected>보통 (30단계)</option>
        <option value="50">최고 (50단계)</option>
      </select>
    </div>
    <img id="cloth-preview" class="thumb" alt="고른 옷" hidden>
  </section>

  <button id="open" class="primary wide" disabled>가상 착용(예시)</button>
  <p class="note">옷 사진을 고르면 버튼이 켜져요. 카메라가 안 되면 창 안에서 사진을 골라도 돼요.</p>
</main>

<dialog id="cam" aria-label="가상 착용">
  <div class="top"><strong>가상 착용(예시)</strong><button id="close">닫기</button></div>
  <div class="stage">
    <video id="video" playsinline muted></video>
    <img id="result" alt="가상 착용 결과" hidden>
    <div id="count" class="count" hidden></div>
    <div id="busy" class="busy" hidden></div>
    <div id="nocam" class="busy" hidden>카메라를 열지 못했어요.<br>아래 '사진 고르기'로 내 사진을 골라 주세요.</div>
  </div>
  <p id="hint" class="note">정면을 보고, 팔을 몸에서 살짝 떼고, 허리 아래까지 보이게 서 주세요.</p>
  <p id="meta" class="note" hidden></p>
  <p id="err" class="note err" hidden></p>
  <button id="shoot" class="primary full">3초 뒤 찍기</button>
  <button id="again" class="primary full" hidden>다시 찍기</button>
  <div class="pair">
    <button id="flip">카메라 전환</button>
    <label class="btn" id="pick-label">사진 고르기<input id="pick" type="file" accept="image/*" hidden></label>
  </div>
</dialog>

<script>
  const $ = (id) => document.getElementById(id);
  const API = new URLSearchParams(location.search).get("api") || "";
  const sleep = (ms) => new Promise((r) => setTimeout(r, ms));
  let stream = null, facing = "user", cloth = null, run = 0;

  $("cloth-file").onchange = (e) => {
    cloth = e.target.files[0] || null;
    if (cloth) { $("cloth-preview").src = URL.createObjectURL(cloth); $("cloth-preview").hidden = false; }
    $("open").disabled = !cloth;
  };

  // view: live | nocam | count | busy | result | error
  function setView(view) {
    const live = view === "live" || view === "count";
    $("video").hidden = !live;
    $("count").hidden = view !== "count";
    $("busy").hidden = view !== "busy";
    $("nocam").hidden = view !== "nocam";
    $("result").hidden = view !== "result";
    $("meta").hidden = view !== "result";
    $("err").hidden = view !== "error";
    $("hint").hidden = !(live || view === "nocam");
    $("shoot").hidden = view !== "live";
    $("flip").hidden = view !== "live";
    $("pick-label").hidden = !(view === "live" || view === "nocam");
    $("again").hidden = !(view === "result" || view === "error");
  }
  function stopCam() { if (stream) stream.getTracks().forEach((t) => t.stop()); stream = null; }
  async function startCam() {
    stopCam();
    try {
      stream = await navigator.mediaDevices.getUserMedia({ video: { facingMode: facing, width: { ideal: 1280 }, height: { ideal: 1280 } }, audio: false });
      $("video").srcObject = stream;
      $("video").classList.toggle("mirror", facing === "user");
      await $("video").play();
      setView("live");
    } catch (err) { stopCam(); setView("nocam"); }
  }

  $("open").onclick = () => { $("cam").showModal(); startCam(); };
  $("close").onclick = () => $("cam").close();
  $("cam").addEventListener("close", () => { run++; stopCam(); });
  $("flip").onclick = () => { facing = facing === "user" ? "environment" : "user"; startCam(); };
  $("again").onclick = () => startCam();
  $("pick").onchange = (e) => { const f = e.target.files[0]; e.target.value = ""; if (f) { stopCam(); send(f); } };
  $("shoot").onclick = async () => {
    const my = ++run;
    setView("count");
    for (let s = 3; s > 0; s--) { $("count").textContent = s; await sleep(1000); if (my !== run) return; }
    const v = $("video"), c = document.createElement("canvas");
    c.width = v.videoWidth; c.height = v.videoHeight;
    c.getContext("2d").drawImage(v, 0, 0);  // 미리보기만 거울처럼 보이고, 찍힌 사진은 실제 방향 그대로
    stopCam();
    c.toBlob((b) => send(b), "image/jpeg", 0.92);
  };

  // 처리에 1분 넘게 걸릴 수 있어 (터널은 100초에 끊김) 작업을 맡기고 1초마다 상태를 물어본다
  async function errText(r) { const b = await r.json().catch(() => ({})); return b.detail || `서버 오류 (${r.status})`; }
  async function send(person) {
    const my = ++run, t0 = Date.now();
    let ahead = 0;
    const tick = () => {
      const wait = ahead > 0 ? ` (앞에 ${ahead}명 대기)` : "";
      $("busy").textContent = `입혀 보는 중… ${Math.round((Date.now() - t0) / 1000)}초${wait}`;
    };
    tick(); setView("busy");
    const timer = setInterval(tick, 500);
    const fd = new FormData();
    fd.append("person", person, "person.jpg");
    fd.append("cloth", cloth, cloth.name || "cloth.jpg");
    fd.append("cloth_type", $("kind").value);
    fd.append("steps", $("steps").value);
    try {
      const r = await fetch(API + "/api/tryon", { method: "POST", body: fd });
      if (!r.ok) throw new Error(await errText(r));
      const { job } = await r.json();
      let s = null, misses = 0;
      while (true) {
        await sleep(1000);
        if (my !== run) return;
        try {
          const q = await fetch(`${API}/api/tryon/${job}`, { cache: "no-store" });
          if (!q.ok) throw new Error(await errText(q));
          s = await q.json(); misses = 0;
        } catch (e) {
          if (!(e instanceof TypeError) || ++misses >= 5) throw e;   // 잠깐 끊긴 건 다시 물어봄
          continue;
        }
        ahead = s.ahead || 0;
        if (s.status === "done") break;
        if (s.status === "error") throw new Error(s.detail);
      }
      const img = await fetch(`${API}/api/tryon/${job}/image`, { cache: "no-store" });
      if (!img.ok) throw new Error(await errText(img));
      const blob = await img.blob();
      if (my !== run) return;
      $("result").src = URL.createObjectURL(blob);
      $("meta").textContent = `AI가 그린 예시 이미지예요. 실제 사이즈와 핏은 반영되지 않아요. · 서버 ${s.elapsed}초, 전체 ${((Date.now() - t0) / 1000).toFixed(1)}초`;
      setView("result");
    } catch (err) {
      if (my !== run) return;
      $("err").textContent = err instanceof TypeError ? "서버에 연결하지 못했어요. Colab 셀이 켜져 있는지 확인해 주세요." : err.message;
      setView("error");
    } finally { clearInterval(timer); }
  }
</script>
</body>
</html>
'''
print(f'시험 페이지 준비 ({len(TEST_PAGE):,}자)')

In [ ]:
# 11. 가상 착용 서버 켜기
!pip install -q fastapi==0.143.0 uvicorn==0.54.0 python-multipart==0.0.32
import io, threading, uuid, uvicorn
from concurrent.futures import ThreadPoolExecutor
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse, Response

# 한 장에 1분 넘게 걸릴 수 있어서(터널은 100초에 끊김) 작업을 받아 두고, 페이지가 1초마다 상태를 물어보게 한다
api = FastAPI()
api.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
jobs = {}                                   # 작업 id → 상태·결과 (메모리에만, 최근 20개)
gpu = ThreadPoolExecutor(max_workers=1)     # GPU는 한 번에 한 장씩


def run_job(jid, person, cloth, cloth_type, steps):
    job = jobs[jid]
    job["status"] = "running"
    try:
        out = try_on(person, cloth, cloth_type, steps=steps)
        buf = io.BytesIO()
        out["result"].save(buf, "JPEG", quality=90)
        job.update(status="done", image=buf.getvalue(), elapsed=round(out["mask_s"] + out["tryon_s"], 1))
    except TryOnError as e:
        job.update(status="error", detail=str(e))
    except Exception as e:
        print("try_on 오류:", repr(e))
        job.update(status="error", detail="입혀 보지 못했어요. 잠시 뒤 다시 해 보세요.")


@api.get("/", response_class=HTMLResponse)
def page():
    return TEST_PAGE


@api.get("/api/health")
def health():
    return {"ok": True, "device": DEVICE}


@api.post("/api/tryon", status_code=202)
def start_tryon(person: UploadFile = File(...), cloth: UploadFile = File(...),
                cloth_type: str = Form("upper"), steps: int = Form(30)):
    if cloth_type not in ("upper", "lower", "overall", "inner", "outer"):
        raise HTTPException(400, "옷 종류가 올바르지 않아요.")
    try:
        p, c = open_image(io.BytesIO(person.file.read())), open_image(io.BytesIO(cloth.file.read()))
    except Exception:
        raise HTTPException(400, "사진을 읽지 못했어요.")
    jid = uuid.uuid4().hex[:12]
    jobs[jid] = {"status": "queued", "created": time.time()}
    for old in list(jobs)[:-20]:
        jobs.pop(old, None)
    gpu.submit(run_job, jid, p, c, cloth_type, max(10, min(50, steps)))
    return {"job": jid}


@api.get("/api/tryon/{jid}")
def tryon_status(jid: str):
    job = jobs.get(jid)
    if job is None:
        raise HTTPException(404, "작업을 찾지 못했어요. 다시 찍어 주세요.")
    ahead = sum(1 for j in list(jobs.values()) if j["status"] in ("queued", "running") and j["created"] < job["created"])
    return {**{k: v for k, v in job.items() if k != "image"}, "ahead": ahead}


@api.get("/api/tryon/{jid}/image")
def tryon_image(jid: str):
    job = jobs.get(jid)
    if job is None or job["status"] != "done":
        raise HTTPException(404, "결과가 아직 없어요.")
    return Response(job["image"], media_type="image/jpeg", headers={"Cache-Control": "no-store"})


if "server" in globals():
    server.should_exit = True
    time.sleep(2)
server = uvicorn.Server(uvicorn.Config(api, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)
print("서버 켜짐:", server.started)

In [ ]:
# 12. 휴대폰·노트북에서 들어갈 주소 만들기
import re
if not os.path.exists("/content/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared
!pkill -f "cloudflared tunnel" || true
!/content/cloudflared tunnel --url http://127.0.0.1:8000 --no-autoupdate > /content/cloudflared.log 2>&1 &
for _ in range(20):
    time.sleep(1)
    urls = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if urls:
        break
print("시험 페이지 주소:", urls[0] if urls else open("/content/cloudflared.log").read()[-1500:])